In [0]:
import dlt

In [0]:
@dlt.table(
    table_properties={"layer" : "bronze"},
    comment="This is the raw order table read"
)
def orders_bronze():
  df= spark.readStream.table("dbacademy.labuser14322013_1774611990.raw_order")
  return df

In [0]:
@dlt.table(
    table_properties={"layer" : "bronze"},
    comment="This is the raw customer table read",
    name="customer_bronze"
)
def cust_bronze():
  df= spark.table("dbacademy.labuser14322013_1774611990.raw_customer")
  return df

In [0]:
@dlt.view(

    comment="This is the raw view after joining order and customer tables",

)
def joined_view():
  df= spark.sql("""
                select * from LIVE.orders_bronze o
                left join LIVE.customer_bronze c
                on o.o_custkey = c.c_custkey
                """)
  return df

In [0]:
import pyspark.sql.functions as F

@dlt.table(
    table_properties={"layer" : "silver"},
    comment="This is the cleaned silver layer  order and customer joined data"
)
def orders_details_silver():
  df= spark.read.table("LIVE.joined_view").withColumn("update_date" , F.current_timestamp())
  return df

In [0]:
@dlt.table(
    table_properties={"layer" : "gold"},
    comment="This is the  gold layer  order and customer joined data"
)
def orders_gold():
  df= spark.read.table("LIVE.orders_details_silver")

  df_agg = df.groupBy("c_mktsegment").agg(F.count("o_orderkey").alias("total_orders") , F.sum("o_totalprice").alias("total_revenue"), F.avg("o_totalprice").alias("avg_revenue")).withColumn("update_timstamp" , F.current_timestamp())
  return df_agg